# 01. Markov Chains & Markov Decision Processes (MDPs)

**The mathematical foundation of sequential decision making and Reinforcement Learning (RL).**

---

## 1. The Markov Property

A stochastic process $\{X_0, X_1, X_2, \dots\}$ satisfies the **Markov Property** (memoryless property) if the conditional probability of future states depends **only on the present state**, not on the history of past states:

$$P(X_{t+1} = s_{t+1} \mid X_t = s_t, X_{t-1} = s_{t-1}, \dots, X_0 = s_0) = P(X_{t+1} = s_{t+1} \mid X_t = s_t)$$

- **"The future is independent of the past given the present."**

---

## 2. Discrete-Time Markov Chains (DTMC)

### Transition Probability Matrix $\mathbf{P}$:
For a system with $K$ states, the transition probability matrix $\mathbf{P} \in \mathbb{R}^{K \times K}$ has entries:
$$P_{ij} = P(X_{t+1} = j \mid X_t = i) \quad \text{where } \sum_{j=1}^K P_{ij} = 1 \text{ (Stochastic Matrix)}$$

- **$n$-Step Transition**: The probability of transitioning from state $i$ to state $j$ in $n$ steps is given by the matrix power $\mathbf{P}^n$!
- **Stationary Distribution $\mathbf{\pi}$**: A probability distribution that satisfies:
  $$\mathbf{\pi} \mathbf{P} = \mathbf{\pi} \quad \iff \quad (\mathbf{P}^T - \mathbf{I})\mathbf{\pi}^T = \mathbf{0}$$
  *(This is the left-eigenvector corresponding to eigenvalue $\lambda = 1$!).*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 3-State Weather Markov Chain: [Sunny (0), Cloudy (1), Rainy (2)]
P = np.array([
    [0.7, 0.2, 0.1],  # From Sunny
    [0.3, 0.4, 0.3],  # From Cloudy
    [0.2, 0.3, 0.5]   # From Rainy
])

print("Transition Matrix P:\n", P)

# 1. 10-step transition probability matrix P^10
P_10 = np.linalg.matrix_power(P, 10)
print("10-Step Transition Matrix P^10:\n", np.round(P_10, 4))

# 2. Compute Stationary Distribution via Eigendecomposition of P^T
eigenvals, eigenvecs = np.linalg.eig(P.T)
# Find eigenvector for eigenvalue closest to 1.0
idx = np.argmin(np.abs(eigenvals - 1.0))
pi = np.real(eigenvecs[:, idx])
pi = pi / np.sum(pi) # Normalize to sum to 1

print("Stationary Distribution pi:", np.round(pi, 4))
print("Verification pi @ P == pi:", np.allclose(pi @ P, pi))


---

## 3. Markov Decision Processes (MDPs)

An **MDP** extends a Markov chain by adding an agent that takes **Actions** and receives **Rewards**.
It is formally defined by a 5-tuple $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$:

1. **$\mathcal{S}$**: State space.
2. **$\mathcal{A}$**: Action space.
3. **$\mathcal{P}(s' \mid s, a)$**: Transition probability to state $s'$ when taking action $a$ in state $s$.
4. **$\mathcal{R}(s, a)$**: Immediate reward received.
5. **$\gamma \in [0, 1)$**: Discount factor for future rewards.

---

## 4. The Bellman Equations (Heart of Reinforcement Learning)

The goal in RL is to find a policy $\pi(a \mid s)$ that maximizes the expected discounted cumulative return $G_t = \sum_{k=0}^\infty \gamma^k R_{t+k+1}$.

### 4.1 Bellman Expectation Equation for State-Value $V^\pi(s)$:
$$V^\pi(s) = \sum_{a \in \mathcal{A}} \pi(a \mid s) \left[ \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) V^\pi(s') \right]$$

### 4.2 Bellman Optimality Equation:
$$V^*(s) = \max_{a \in \mathcal{A}} \left[ \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) V^*(s') \right]$$
$$Q^*(s, a) = \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) \max_{a'} Q^*(s', a')$$

*Solving this fixed-point equation via Dynamic Programming gives the optimal policy in Q-Learning and Value Iteration!*


In [ ]:
# GridWorld Value Iteration (Simplified MDP Solver)
num_states = 4 # [State 0, State 1, State 2, Goal 3]
rewards = np.array([-1.0, -1.0, -1.0, 10.0])
gamma = 0.9

V = np.zeros(num_states)
# Value Iteration Loop
for iteration in range(20):
    V_new = V.copy()
    for s in range(num_states - 1): # Goal is absorbing
        # Actions: Move forward (s+1) or stay (s)
        v_forward = rewards[s+1] + gamma * V[s+1]
        v_stay = rewards[s] + gamma * V[s]
        V_new[s] = max(v_forward, v_stay)
    V = V_new

print("Optimal State Values V*(s) after Value Iteration:")
for s in range(num_states):
    print(f"State {s}: V*(s) = {V[s]:.4f}")


---

## 5. Summary & Key Takeaways

1. **The Markov Property** states that the future depends only on the present state.
2. **Stationary Distribution** $\mathbf{\pi}\mathbf{P} = \mathbf{\pi}$ describes the long-term equilibrium behavior of a Markov chain.
3. **Markov Decision Processes (MDPs)** formalize Reinforcement Learning environments.
4. **Bellman Equations** decompose optimal values recursively into immediate reward + discounted expected future return.
